In [ ]:
#import
import sys
sys.path.append("..")
from topofinger import config
import numpy as np
import matplotlib.pyplot as plt

from tqdm.notebook import tqdm
import os

import pickle as pk
from nilearn import plotting
import pandas as pd

import sys

from topofinger.hcp_IO import load_ts_schaefer300_dict, load_fc_cortical
from scipy.spatial.distance import squareform
import random

from topofinger.utils import threshold_matrix

from topofinger.utils_neuromaps_brain_multibrain import normal_view as normal_view_multibrain
from topofinger.cycle_synergy_utils import get_subject_synergy_metrics # Import the new function


%load_ext autoreload
%autoreload 2

#from visualize import threshold_adj_matrix

#paths
ts_folder = config.HCP_1200 + "/NoGSR_LowPassOnly/ts/"
fc_folder = config.HCP_1200 + "/NoGSR_LowPassOnly/fc/whole/"
# PhiR synergy and MMI redundancy, session REST1_LR, from high-pass-only time series (scripts/08_synergy_phiR.py)
syn_file = config.HCP_1200 + "/NoGSR_HighPassOnly/info/phiR_s0.npz"
# H1 cycle representatives, 300 cortical regions, session REST1_LR (scripts/01b_representatives_cortex_1200.jl)
rep_folder = config.HCP_1200 + "/NoGSR_HighPassOnly/scaffolds/whole_cortex/representatives/"
figure_folder = config.FIGURES_RAW + "/fullhcp/fig4/"

## Panel A

In [ ]:
#doesn't work on the server at the moment


# Set the DISPLAY variable
os.environ['DISPLAY'] = ':0'

#es_sig = np.random.random((300, ))
es_sig = np.random.random(300)
normal_view_multibrain(es_sig, edges=True, graymap_rev=True, exp_form=True, cmap='Blues',
                       alpha_graymap=0.99, center_cbar=True, cbar_pad=0, parcellation=300,
                       fsize_cbar=12)

In [ ]:
## LOAD TS DATA ##

ts_data = load_ts_schaefer300_dict(ts_folder)
print('The dataset is composed of ' + str(len(ts_data.keys())) + ' subjects')
print('Each subject performed ' + str(len(ts_data['103414'].keys())) + ' trials')
print('Each trial contains ' + str(ts_data['103414'][0].shape[1]) +' time series (number of brain regions)')
print('The lenght of each time serie is ' + str(ts_data['103414'][0].shape[0]) + ' time points')

In [ ]:
subjects = list(ts_data.keys())[:3]

i=1
for subject in subjects:
    plt.figure(figsize=(4,1))
    plt.plot(ts_data[subject][0].T[9,:100], c='k')
    plt.axis('off')
    plt.savefig(figure_folder + f'fig4A_timeseries{i}.png', transparent=True, dpi=300)
    i += 1
    plt.show()



# plt.figure(figsize=(4,1))
# plt.plot(ts_data[subject][1].T[9,:100], c='violet', alpha=0.7)
# plt.axis('off')

# plt.savefig('fig1_timeseries3.png', transparent=True, dpi=300)
# plt.show()

In [ ]:
subject = '100408'

## LOAD FC DATA (300 cortical regions) ##
fc_hp = load_fc_cortical(config.HCP_1200 + "/NoGSR_HighPassOnly")
print(f"FC for {len(fc_hp)} subjects, {fc_hp[subject][0].shape[0]} regions")

fc_example = fc_hp[subject][0].copy()
np.fill_diagonal(fc_example, 0)
plotting.plot_matrix(fc_example, colorbar=False, cmap='Blues')
plt.axis('off')
plt.savefig(figure_folder + 'fig4A_FC.png', transparent=True, dpi=300)
plt.show()


In [ ]:
## Load PhiR synergy and MMI redundancy (session REST1_LR, condensed upper triangles)
phi = np.load(syn_file)
phi_subjects = [str(s) for s in phi['subjects']]
syn = {s: squareform(phi['syn'][k]) for k, s in enumerate(phi_subjects)}
red = {s: squareform(phi['red'][k]) for k, s in enumerate(phi_subjects)}
print(f"Synergy and redundancy for {len(phi_subjects)} subjects")


In [ ]:
plotting.plot_matrix(red[subject], colorbar=False, cmap='Greens', vmin=0, vmax=np.percentile(red[subject], 99))
plt.axis('off')

plt.savefig(figure_folder + 'fig4A_red.png', transparent=True, dpi=300)
plt.show()


In [ ]:
plotting.plot_matrix(syn[subject], colorbar=False, cmap='Purples', vmin=0, vmax=np.percentile(syn[subject], 99))
plt.axis('off')

plt.savefig(figure_folder + 'fig4A_syn.png', transparent=True, dpi=300)
plt.show()


In [ ]:
# FC vs synergy vs redundancy, session REST1_LR, upper triangles (same order as the condensed synergy)
iu = np.triu_indices(config.N_CORTICAL, 1)
FC = np.array([fc_hp[s][0][iu] for s in phi_subjects])
SYN, RED = phi['syn'].astype(float), phi['red'].astype(float)

# Average across subjects
avg_fc_flat, avg_syn_flat, avg_red_flat = FC.mean(0), SYN.mean(0), RED.mean(0)
print("Correlations (Averaged across subjects):")
print(f"FC vs Synergy: {np.corrcoef(avg_fc_flat, avg_syn_flat)[0, 1]:.4f}")
print(f"FC vs Redundancy: {np.corrcoef(avg_fc_flat, avg_red_flat)[0, 1]:.4f}")
print(f"Synergy vs Redundancy: {np.corrcoef(avg_syn_flat, avg_red_flat)[0, 1]:.4f}")

# Subject level
corr_fc_syn_subjects = np.array([np.corrcoef(f, s)[0, 1] for f, s in zip(FC, SYN)])
corr_fc_red_subjects = np.array([np.corrcoef(f, r)[0, 1] for f, r in zip(FC, RED)])
corr_syn_red_subjects = np.array([np.corrcoef(s, r)[0, 1] for s, r in zip(SYN, RED)])
print("\nCorrelations (Subject Level):")
print(f"FC vs Synergy: Mean = {np.nanmean(corr_fc_syn_subjects):.4f}, Std = {np.nanstd(corr_fc_syn_subjects):.4f}")
print(f"FC vs Redundancy: Mean = {np.nanmean(corr_fc_red_subjects):.4f}, Std = {np.nanstd(corr_fc_red_subjects):.4f}")
print(f"Synergy vs Redundancy: Mean = {np.nanmean(corr_syn_red_subjects):.4f}, Std = {np.nanstd(corr_syn_red_subjects):.4f}")


In [ ]:
# Define colors for consistency
fc_color = 'slateblue'
syn_color = 'olive'
red_color = 'darkcyan'

In [ ]:
# Plot 1: FC vs Synergy (clean)
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(1, 1, figsize=(5, 5))

# FC vs Synergy scatter plot
ax.scatter(avg_fc_flat, avg_syn_flat, alpha=0.6, color=fc_color, s=1)
ax.set_xticks([])
ax.set_yticks([])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
# ax.spines['bottom'].set_visible(False)
# ax.spines['left'].set_visible(False)

plt.tight_layout()
plt.savefig(figure_folder + 'fig4B_FC_vs_Synergy_clean.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 2: FC vs Redundancy (clean)
fig, ax = plt.subplots(1, 1, figsize=(5, 5))

# FC vs Redundancy scatter plot
ax.scatter(avg_fc_flat, avg_red_flat, alpha=0.6, color=red_color, s=1)
ax.set_xticks([])
ax.set_yticks([])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
# ax.spines['bottom'].set_visible(False)
# ax.spines['left'].set_visible(False)

plt.tight_layout()
plt.savefig(figure_folder + 'fig4B_FC_vs_Redundancy_clean.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 3: Synergy vs Redundancy (clean)
fig, ax = plt.subplots(1, 1, figsize=(5, 5))

# Synergy vs Redundancy scatter plot
ax.scatter(avg_syn_flat, avg_red_flat, alpha=0.6, color=syn_color, s=1)
ax.set_xticks([])
ax.set_yticks([])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
# ax.spines['bottom'].set_visible(False)
# ax.spines['left'].set_visible(False)

plt.tight_layout()
plt.savefig(figure_folder + 'fig4B_Synergy_vs_Redundancy_clean.png', dpi=300, bbox_inches='tight')
plt.show()

## Panel C

In [ ]:
# Panel C: synergy of cycle borders, internal links and null models (session REST1_LR)
# Border links = cycle edges, internal links = missing chords of each H1 representative (1-based indices)
cycles = {}
for s in phi_subjects:
    with open(rep_folder + f"{s}_s0.pck", "rb") as f:
        cycles[s] = pk.load(f)
print(f"Representatives for {len(cycles)} subjects")


In [ ]:
np.random.seed(0); random.seed(0)
all_subject_metrics = {s: get_subject_synergy_metrics(cycles[s], syn[s].astype(float), fc_hp[s][0])
                       for s in tqdm(phi_subjects)}
metrics_df = pd.DataFrame.from_dict(all_subject_metrics, orient='index')
print(metrics_df.describe().round(4))


## Boxplots

In [ ]:
# Define the file path for the new metrics dataframe
metrics_file = config.RESULTS + "/synergy_metrics.pck"

# Save the dataframe
metrics_df.to_pickle(metrics_file)
print(f"Saved synergy metrics to {metrics_file}")

In [ ]:
# Load the saved metrics dataframe
metrics_file = config.RESULTS + "/synergy_metrics.pck"
loaded_metrics_df = pd.read_pickle(metrics_file)
print("Loaded synergy metrics:")
print(loaded_metrics_df.head())
print("\nColumn names:")
print(loaded_metrics_df.columns.tolist())
print("\nData types:")
print(loaded_metrics_df.dtypes)
print("\nShape:")
print(loaded_metrics_df.shape)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D  # <- added for legend markers

# Column mapping
column_names = ['border', 'internal', 'random', 'random_fc_matched', 'random_cycle_internal']
labels = ['Border', 'Internal', 'Random', 'Random (FC-Matched)', 'Internal Random Cycles']

# Prepare data for boxplot
plot_data = [loaded_metrics_df[col].dropna() for col in column_names]

# Statistical tests: Compare Internal vs all others
internal_data = loaded_metrics_df['internal'].dropna()
comparisons = []
p_values = []

for i, (col, label) in enumerate(zip(column_names, labels)):
    if label != 'Internal':
        other_data = loaded_metrics_df[col].dropna()
        
        # Use paired t-test (paired, parametric) as originally used
        try:
            stat, p_val = stats.ttest_rel(internal_data, other_data, alternative='two-sided')
            comparisons.append(('Internal', label))
            p_values.append(p_val)
        except Exception as e:
            print(f"Error comparing Internal vs {label}: {e}")

# Apply FDR correction
if len(p_values) > 0:
    fdr_results = multipletests(p_values, alpha=0.05, method='fdr_bh')
    fdr_corrected_p = fdr_results[1]
    fdr_significant = fdr_results[0]
    
    significant_pairs = [comp for i, comp in enumerate(comparisons) if fdr_significant[i]]
    
    print(f"Significant comparisons (FDR-corrected): {len(significant_pairs)}")
    for i, comp in enumerate(comparisons):
        status = "*" if fdr_significant[i] else "ns"
        print(f"  {comp[0]} vs {comp[1]}: p = {fdr_corrected_p[i]:.3e} {status}")
else:
    significant_pairs = []
    fdr_corrected_p = []

# Create the plot
mpl.rcParams["axes.spines.right"] = False
mpl.rcParams["axes.spines.top"] = False
fig, ax = plt.subplots(figsize=(12, 9))

# Define custom colors: Border = violet, Internal = red, others = blue
custom_colors = ['purple', 'red', 'royalblue', 'royalblue', 'royalblue']

# Create transparent boxplot
box = ax.boxplot(plot_data, tick_labels=labels, patch_artist=True, showfliers=False, 
                 medianprops=dict(color="black", linewidth=1.5))

# Apply custom colors with transparency to boxplots
for patch, color in zip(box['boxes'], custom_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.4)  # Make boxplots transparent
    patch.set_edgecolor('black')
    patch.set_linewidth(1)

# Prepare data for stripplot overlay
stripplot_data = []
stripplot_positions = []
stripplot_colors = []

for i, (data, color) in enumerate(zip(plot_data, custom_colors)):
    stripplot_data.extend(data)
    stripplot_positions.extend([i + 1] * len(data))  # Position 1-indexed for boxplot
    stripplot_colors.extend([color] * len(data))

# Add stripplot overlay
for i, (data, color) in enumerate(zip(plot_data, custom_colors)):
    x_pos = np.random.normal(i + 1, 0.01, len(data))  # Add jitter
    ax.scatter(x_pos, data, color=color, alpha=0.6, s=20, edgecolors='white', linewidth=0.5)

# Add significance annotations
if significant_pairs:
    # Get positions for Internal (position 2, 1-indexed)
    internal_pos = 2
    y_max = max([max(data) for data in plot_data])
    y_range = y_max - min([min(data) for data in plot_data])
    
    annotation_height = y_max + 0.05 * y_range
    height_increment = 0.12 * y_range
    
    annotation_count = 0
    for i, (comp, p_val) in enumerate(zip(comparisons, fdr_corrected_p)):
        if comp in significant_pairs:
            # Find position of the other group
            other_label = comp[1]
            other_pos = labels.index(other_label) + 1
            
            current_height = annotation_height + annotation_count * height_increment
            
            # Draw annotation lines
            ax.plot([internal_pos, other_pos], [current_height, current_height], 'k-', linewidth=1)
            ax.plot([internal_pos, internal_pos], [current_height - 0.01 * y_range, current_height], 'k-', linewidth=1)
            ax.plot([other_pos, other_pos], [current_height - 0.01 * y_range, current_height], 'k-', linewidth=1)
            
            # Add significance stars
            if p_val < 0.001:
                p_text = '***'
            elif p_val < 0.01:
                p_text = '**'
            elif p_val < 0.05:
                p_text = '*'
            else:
                p_text = 'ns'
                
            ax.text((internal_pos + other_pos) / 2, current_height + 0.02 * y_range, p_text, 
                   ha='center', va='bottom', fontsize=12)
            
            annotation_count += 1
    
    # Add legend for p-value mapping (stars -> p ranges)
    legend_handles = [
        Line2D([0], [0], marker='$***$', color='k', linestyle='None', markersize=16),
        ]
    legend_labels = ['p < 0.001']
    ax.legend(legend_handles, legend_labels, title='Significance', loc='best',
              fontsize=16, title_fontsize=16, bbox_to_anchor=(1.0, 0.7))
    
    # Adjust y-axis limits
    ax.set_ylim(ax.get_ylim()[0], annotation_height + annotation_count * height_increment + 0.1 * y_range)

ax.set_ylabel('Synergy', fontsize=22)
ax.tick_params(axis='x', rotation=30, labelsize=18)
ax.tick_params(axis='y', labelsize=18)
plt.tight_layout()
plt.savefig(config.FIGURES_RAW + '/final/fig4/fig4C.png', dpi=300, bbox_inches="tight")
plt.show()

# Summary statistics
print("\nSummary Statistics:")
for i, (col, label) in enumerate(zip(column_names, labels)):
    data = plot_data[i]
    print(f"{label}: {np.mean(data):.3f} ± {np.std(data):.3f}")

In [ ]:
# Comprehensive Statistical Analysis: All Pairwise Comparisons
from itertools import combinations
from scipy.stats import shapiro, normaltest
import warnings
warnings.filterwarnings('ignore')

# Test functions
def permutation_test(x, y, n_permutations=10000):
    """Custom permutation test"""
    observed_diff = np.mean(x) - np.mean(y)
    combined = np.concatenate([x, y])
    count = 0
    for _ in range(n_permutations):
        np.random.shuffle(combined)
        perm_diff = np.mean(combined[:len(x)]) - np.mean(combined[len(x):])
        if abs(perm_diff) >= abs(observed_diff):
            count += 1
    return count / n_permutations

def run_statistical_tests(data1, data2, label1, label2):
    """Run multiple statistical tests between two distributions"""
    results = {}
    
    # Basic statistics
    results['mean_diff'] = np.mean(data1) - np.mean(data2)
    results['effect_size_cohens_d'] = (np.mean(data1) - np.mean(data2)) / np.sqrt((np.var(data1, ddof=1) + np.var(data2, ddof=1)) / 2)
    
    # Test normality
    try:
        _, p_norm1 = shapiro(data1)
        _, p_norm2 = shapiro(data2)
        results['normal_dist'] = (p_norm1 > 0.05) and (p_norm2 > 0.05)
    except:
        results['normal_dist'] = False
    
    # Parametric tests (if normal)
    if results['normal_dist'] and len(data1) == len(data2):
        try:
            stat, p_val = stats.ttest_rel(data1, data2)
            results['paired_ttest'] = p_val
        except:
            results['paired_ttest'] = np.nan
    else:
        results['paired_ttest'] = np.nan
    
    try:
        stat, p_val = stats.ttest_ind(data1, data2)
        results['independent_ttest'] = p_val
    except:
        results['independent_ttest'] = np.nan
    
    # Non-parametric tests
    if len(data1) == len(data2):
        try:
            stat, p_val = stats.wilcoxon(data1, data2, alternative='two-sided')
            results['wilcoxon'] = p_val
        except:
            results['wilcoxon'] = np.nan
    else:
        results['wilcoxon'] = np.nan
    
    try:
        stat, p_val = stats.mannwhitneyu(data1, data2, alternative='two-sided')
        results['mannwhitney'] = p_val
    except:
        results['mannwhitney'] = np.nan
    
    # Permutation test
    try:
        results['permutation'] = permutation_test(data1, data2)
    except:
        results['permutation'] = np.nan
    
    # Kolmogorov-Smirnov test (for distribution differences)
    try:
        stat, p_val = stats.ks_2samp(data1, data2)
        results['ks_test'] = p_val
    except:
        results['ks_test'] = np.nan
    
    return results

# Prepare data
column_names = ['border', 'internal', 'random', 'random_fc_matched', 'random_cycle_internal']
labels = ['Border', 'Internal', 'Random', 'Random (FC Match)', 'Random Cycle Internal']
plot_data = [loaded_metrics_df[col].dropna() for col in column_names]

# Run all pairwise comparisons
all_comparisons = list(combinations(range(len(labels)), 2))
results_summary = []

print("=" * 80)
print("COMPREHENSIVE STATISTICAL ANALYSIS: ALL PAIRWISE COMPARISONS")
print("=" * 80)

for i, j in all_comparisons:
    label1, label2 = labels[i], labels[j]
    data1, data2 = plot_data[i], plot_data[j]
    
    print(f"\n{label1} vs {label2}:")
    print("-" * 50)
    
    # Basic statistics
    print(f"Sample sizes: {len(data1)} vs {len(data2)}")
    print(f"Means: {np.mean(data1):.4f} vs {np.mean(data2):.4f}")
    print(f"Std devs: {np.std(data1):.4f} vs {np.std(data2):.4f}")
    
    # Run tests
    test_results = run_statistical_tests(data1, data2, label1, label2)
    
    print(f"Mean difference: {test_results['mean_diff']:.4f}")
    print(f"Cohen's d: {test_results['effect_size_cohens_d']:.4f}")
    print(f"Normal distributions: {test_results['normal_dist']}")
    
    print("\nTest Results (p-values):")
    if not np.isnan(test_results['paired_ttest']):
        print(f"  Paired t-test: {test_results['paired_ttest']:.4f}")
    print(f"  Independent t-test: {test_results['independent_ttest']:.4f}")
    if not np.isnan(test_results['wilcoxon']):
        print(f"  Wilcoxon signed-rank: {test_results['wilcoxon']:.4f}")
    print(f"  Mann-Whitney U: {test_results['mannwhitney']:.4f}")
    print(f"  Permutation test: {test_results['permutation']:.4f}")
    print(f"  Kolmogorov-Smirnov: {test_results['ks_test']:.4f}")
    
    # Store for summary
    test_results['comparison'] = f"{label1} vs {label2}"
    results_summary.append(test_results)

# Convert to DataFrame for easy analysis
results_df = pd.DataFrame(results_summary)

print("\n" + "=" * 80)
print("SUMMARY TABLE: ALL COMPARISONS")
print("=" * 80)

# Create summary table
summary_table = results_df[['comparison', 'mean_diff', 'effect_size_cohens_d', 
                           'independent_ttest', 'mannwhitney', 'permutation', 'ks_test']].copy()
summary_table.columns = ['Comparison', 'Mean Diff', "Cohen's d", 't-test', 'Mann-Whitney', 'Permutation', 'KS-test']

# Round for display
for col in ['Mean Diff', "Cohen's d", 't-test', 'Mann-Whitney', 'Permutation', 'KS-test']:
    summary_table[col] = summary_table[col].round(4)

print(summary_table.to_string(index=False))

# Apply multiple comparisons correction
print("\n" + "=" * 80)
print("MULTIPLE COMPARISONS CORRECTION")
print("=" * 80)

test_names = ['independent_ttest', 'mannwhitney', 'permutation', 'ks_test']

for test_name in test_names:
    p_values = results_df[test_name].dropna().values
    if len(p_values) > 0:
        # FDR correction
        fdr_results = multipletests(p_values, alpha=0.05, method='fdr_bh')
        # Bonferroni correction
        bonf_results = multipletests(p_values, alpha=0.05, method='bonferroni')
        
        print(f"\n{test_name.upper()} - Significant comparisons:")
        print(f"  Raw p < 0.05: {np.sum(p_values < 0.05)}")
        print(f"  FDR corrected: {np.sum(fdr_results[0])}")
        print(f"  Bonferroni corrected: {np.sum(bonf_results[0])}")
        
        # Show which comparisons are significant after correction
        valid_indices = results_df[test_name].notna()
        results_df.loc[valid_indices, f'{test_name}_fdr_significant'] = fdr_results[0]
        results_df.loc[valid_indices, f'{test_name}_bonf_significant'] = bonf_results[0]

# Summary of effect sizes
print("\n" + "=" * 80)
print("EFFECT SIZE SUMMARY")
print("=" * 80)

effect_sizes = results_df['effect_size_cohens_d'].abs()
print(f"Mean absolute effect size: {np.mean(effect_sizes):.3f}")
print(f"Largest effect size: {np.max(effect_sizes):.3f} ({results_df.loc[effect_sizes.idxmax(), 'comparison']})")
print(f"Smallest effect size: {np.min(effect_sizes):.3f} ({results_df.loc[effect_sizes.idxmin(), 'comparison']})")

# Effect size interpretation
large_effects = results_df[effect_sizes > 0.8]['comparison'].tolist()
medium_effects = results_df[(effect_sizes > 0.5) & (effect_sizes <= 0.8)]['comparison'].tolist()
small_effects = results_df[(effect_sizes > 0.2) & (effect_sizes <= 0.5)]['comparison'].tolist()

print(f"\nLarge effects (|d| > 0.8): {len(large_effects)}")
for comp in large_effects:
    print(f"  {comp}")
    
print(f"\nMedium effects (0.5 < |d| ≤ 0.8): {len(medium_effects)}")
for comp in medium_effects:
    print(f"  {comp}")

print(f"\nSmall effects (0.2 < |d| ≤ 0.5): {len(small_effects)}")
for comp in small_effects:
    print(f"  {comp}")